## Calculade a clasificación do circuíto aplicando as normas de puntuación do regulamento.

## Configuration

In [8]:
# %%
import os
import re
import hmac
import hashlib
import unicodedata
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

from src.scraping import Scraper, EdicionConfig, TAM_PAGINA

# ==============================================================================
# 1. CONFIGURACIÓN DE REGLAS (Separada de la lógica de cálculo)
# ==============================================================================
load_dotenv()
CLAVE_SECRETA = os.getenv("HMAC_SECRET_KEY", "").encode("utf-8")

REGLAS_CIRCUITO = {
    "min_pruebas_clasificar": 5,           # Art. 9 (pág. 11)
    "max_pruebas_computables": 5,          # Art. 9 (pág. 11)
    "columna_puntos": "puesto_por_sexo",   # Cambiar a 'puesto_general' si la web usa general absoluta
    "valor_inscrito": "inscrito_circuito", # Art. 4.A y Art. 9
    "estados_tiempo_validos": ["valido", "revisar"]  # Según criterio de limpieza.py
}

# ==============================================================================
# 2. MOTOR DE CÁLCULO SEGÚN REGLAMENTO
# ==============================================================================
def calcular_clasificacion_circuito(df_part: pd.DataFrame, reglas: dict) -> pd.DataFrame:
    df = df_part.copy()
    
    # Filtrar inscritos al circuito y participaciones con tiempo válido
    df = df[
        (df["tipo_inscripcion"] == reglas["valor_inscrito"]) &
        (df["estado_tiempo"].isin(reglas["estados_tiempo_validos"]))
    ].copy()
    
    col_pts = reglas["columna_puntos"]
    df["puntos_carrera"] = pd.to_numeric(df[col_pts], errors="coerce")
    df = df.dropna(subset=["puntos_carrera"])
    
    # Ordenar resultados de cada corredor de mejor (menor) a peor (mayor) puntuación
    df = df.sort_values(["persoa_id", "puntos_carrera", "id_carrera"])
    df["rank_carrera_persona"] = df.groupby("persoa_id").cumcount() + 1
    
    # Quedarnos con las N mejores pruebas (máximo 5)
    mejores = df[df["rank_carrera_persona"] <= reglas["max_pruebas_computables"]]
    
    # Resumen total de carreras finalizadas y tupla ordenada de puestos para desempates
    resumen = df.groupby(["persoa_id", "sexo", "categoria"], as_index=False).agg(
        pruebas_finalizadas=("id_carrera", "count"),
        secuencia_puestos=("puntos_carrera", lambda s: tuple(int(x) for x in sorted(s)))
    )
    
    suma_puntos = (
        mejores.groupby("persoa_id", as_index=False)["puntos_carrera"]
        .sum()
        .rename(columns={"puntos_carrera": "puntos_calc"})
    )
    
    clasif = resumen.merge(suma_puntos, on="persoa_id", how="inner")
    
    # Exigir el mínimo de 5 pruebas finalizadas
    clasif = clasif[clasif["pruebas_finalizadas"] >= reglas["min_pruebas_clasificar"]].copy()
    
    # Ordenar aplicando desempates del Art. 9:
    # 1º Menor suma de puntos | 2º Más pruebas finalizadas | 3º Mejor secuencia de puestos
    clasif["neg_pruebas"] = -clasif["pruebas_finalizadas"]
    clasif = clasif.sort_values(
        by=["sexo", "puntos_calc", "neg_pruebas", "secuencia_puestos"]
    ).drop(columns=["neg_pruebas"])
    
    clasif["puesto_sexo_calc"] = clasif.groupby("sexo").cumcount() + 1
    clasif["puesto_cat_calc"] = clasif.groupby(["sexo", "categoria"]).cumcount() + 1
    
    return clasif

df_part = pd.read_csv("../datos/clean/participacions.csv")
df_propia = calcular_clasificacion_circuito(df_part, REGLAS_CIRCUITO)

print(f"Total personas clasificadas (>=5 pruebas): {len(df_propia)}")
print(df_propia["sexo"].value_counts())

# ==============================================================================
# 3. PRUEBA MANUAL Y CASOS BORDE (Exigido en guía O4)
# ==============================================================================
print("\n--- PRUEBA MANUAL: 1.º Clasificado Masculino ---")
id_top1 = df_propia[df_propia["sexo"] == "M"].iloc[0]["persoa_id"]
display(df_part[df_part["persoa_id"] == id_top1][["persoa_id", "id_carrera", "puesto_general", "puesto_por_sexo", "estado_tiempo"]])
display(df_propia[df_propia["persoa_id"] == id_top1])

print("\n--- CASOS BORDE: Empates a puntos y categorías especiales ('NO') ---")
empates = df_propia[df_propia.duplicated(subset=["sexo", "puntos_calc"], keep=False)]
print(f"Corredores empatados a puntos dentro del mismo sexo: {len(empates)}")
if not empates.empty:
    display(empates.head(6))

cat_especiales = df_propia[df_propia["categoria"] == "NO"]
print(f"Clasificados en categoría especial 'NO': {len(cat_especiales)}")

ModuleNotFoundError: No module named 'src'

In [9]:
# %%
# ==============================================================================
# 4. DESCARGA Y PSEUDONIMIZACIÓN DE LA CLASIFICACIÓN OFICIAL (/clasificacion/157)
# ==============================================================================
cfg_clasif = EdicionConfig(
    num_edicion=63,
    idioma="es",
    patron_evento=r"evento-(\d+)",
    prefijo_carrera="clasif_157",
    raw_dir=Path("../datos/raw/clasificacion_oficial"),
    cache_eventos="eventos_ed63.html"
)
scraper = Scraper(cfg_clasif)

def normalizar_nombre(texto: str) -> str:
    if pd.isna(texto):
        return ""
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("utf-8")
    return " ".join(texto.upper().split())

def generar_hmac(nombre_norm: str) -> str:
    return hmac.new(CLAVE_SECRETA, nombre_norm.encode("utf-8"), hashlib.sha256).hexdigest()

def obtener_clasificacion_oficial_segura(url_relativa="/es/web/resultado/edicion-63/clasificacion/157") -> pd.DataFrame:
    todas_filas, previos, pagina, encabezados = [], set(), 1, []

    while True:
        nombre_raw = f"clasif_157_pag_{pagina:02d}.html"
        html = scraper.descargar_pagina(f"{cfg_clasif.base_url}{url_relativa}?page={pagina}", nombre_raw)
        soup = BeautifulSoup(html, "lxml")
        tabla = soup.find("table")
        if not tabla:
            (cfg_clasif.raw_dir / nombre_raw).unlink(missing_ok=True)
            break

        if not encabezados:
            encabezados = [th.get_text(" ", strip=True) for th in tabla.find_all("th")]

        tbody = tabla.find("tbody")
        trs = tbody.find_all("tr") if tbody else tabla.find_all("tr")[1:]
        filas_pag = []

        for tr in trs:
            tds = [td.get_text(" ", strip=True) for td in tr.find_all("td")]
            if len(tds) < 4:
                continue
            fila = dict(zip(encabezados, tds)) if len(encabezados) == len(tds) else {f"col_{i}": v for i, v in enumerate(tds)}
            filas_pag.append(fila)

        if not filas_pag:
            (cfg_clasif.raw_dir / nombre_raw).unlink(missing_ok=True)
            break

        huella = {tuple(f.values()) for f in filas_pag}
        if huella == previos:
            (cfg_clasif.raw_dir / nombre_raw).unlink(missing_ok=True)
            break

        previos = huella
        todas_filas.extend(filas_pag)
        if len(filas_pag) < TAM_PAGINA:
            break
        pagina += 1

    df_raw = pd.DataFrame(todas_filas)
    print(f"Descarga completada: {len(df_raw)} filas en {pagina} páginas. Columnas: {encabezados}")

    # Pseudonimización inmediata y borrado de columnas sensibles
    cols_nom = [c for c in df_raw.columns if any(k in c.lower() for k in ["nombre", "nome", "apellido", "apelido", "atleta"])]
    df_raw["persoa_id"] = (
        df_raw[cols_nom].fillna("").agg(" ".join, axis=1)
        .apply(normalizar_nombre)
        .apply(generar_hmac)
    )
    cols_sensibles = cols_nom + [c for c in df_raw.columns if "dorsal" in c.lower()]
    df_seguro = df_raw.drop(columns=cols_sensibles, errors="ignore").copy()
    del df_raw
    return df_seguro

df_oficial = obtener_clasificacion_oficial_segura()

# ==============================================================================
# 5. CRUCE PERSONA A PERSONA Y CLASIFICACIÓN DE DISCREPANCIAS
# ==============================================================================
# Detección automática de columnas de puntos y puesto en la tabla oficial
col_pts_of = next((c for c in df_oficial.columns if "punt" in c.lower() or "total" in c.lower()), None)
col_pos_of = next((c for c in df_oficial.columns if "pos" in c.lower() or "puesto" in c.lower() or "posto" in c.lower()), None)

if col_pts_of:
    df_oficial["puntos_oficial"] = pd.to_numeric(df_oficial[col_pts_of], errors="coerce")
if col_pos_of:
    df_oficial["puesto_oficial"] = pd.to_numeric(df_oficial[col_pos_of].astype(str).str.extract(r"(\d+)")[0], errors="coerce")

comp = pd.merge(df_propia, df_oficial, on="persoa_id", how="outer", indicator=True)

if "puntos_oficial" in comp.columns:
    comp["diff_puntos"] = comp["puntos_calc"] - comp["puntos_oficial"]
if "puesto_oficial" in comp.columns:
    comp["diff_puesto"] = comp["puesto_sexo_calc"] - comp["puesto_oficial"]

# Clasificar el motivo de cada discrepancia según guía O4
def clasificar_discrepancia(row):
    if row["_merge"] == "both":
        if row.get("diff_puntos", 0) == 0 and row.get("diff_puesto", 0) == 0:
            return "Coincidencia exacta"
        elif row.get("diff_puntos", 0) != 0:
            return "Ambigüedad reglamento / criterio puntos"
        else:
            return "Diferencia de desempate / orden"
    elif row["_merge"] == "left_only":
        return "Solo en cálculo propio (posible cambio de nombre/HMAC o filtro web)"
    else:
        return "Solo en web oficial (posible <5 pruebas válidas en datos limpios)"

comp["diagnostico"] = comp.apply(clasificar_discrepancia, axis=1)

print("\n--- RESUMEN DE AUDITORÍA (Opcional 04) ---")
display(comp["diagnostico"].value_counts().to_frame("num_personas"))

# Exportar tabla de discrepancias sin identificadores directos a output/
os.makedirs("../output", exist_ok=True)
cols_export = [c for c in ["persoa_id", "sexo", "categoria", "pruebas_finalizadas", "puntos_calc", "puntos_oficial", "diff_puntos", "puesto_sexo_calc", "puesto_oficial", "diagnostico"] if c in comp.columns]
comp[comp["diagnostico"] != "Coincidencia exacta"][cols_export].to_csv("../output/discrepancias_clasificacion_o4.csv", index=False)
print("✅ Tabla de discrepancias exportada de forma segura en ../output/discrepancias_clasificacion_o4.csv")

NameError: name 'EdicionConfig' is not defined